# Latent Knowledge Validation

Validate market intuitions encoded as features:
- Test each hypothesis against historical data
- Compute IC (Information Coefficient) for each
- Analyze regime-conditional performance
- Generate data acquisition recommendations

Goal: Determine which market intuitions are actually predictive.

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
from scipy import stats

from src.data.feature_engineering import (
    LatentKnowledgeFeatures,
    EnhancedTechnicalFeatures,
    AlternativeDataFeatures,
)
from src.data.features import FeatureEngine
from src.data.sources.yahoo_finance import YahooFinanceDataSource

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Define Hypotheses

In [ ]:
# Get all latent knowledge hypotheses
hypotheses = LatentKnowledgeFeatures.get_hypotheses()

print(f"Total hypotheses to test: {len(hypotheses)}")
print("\n" + "="*60)

for name, description in hypotheses.items():
    print(f"\n{name}:")
    print(f"  {description}")

## 2. Load Price Data

In [ ]:
# Load price data for testing
data_source = YahooFinanceDataSource()

# Test symbols
TEST_SYMBOLS = ['SPY', 'QQQ', 'XLF', 'XLE', 'SMH']

price_data = {}
for symbol in TEST_SYMBOLS:
    try:
        df = data_source.get_historical_data(symbol, period='2y')
        price_data[symbol] = df
        print(f"Loaded {symbol}: {len(df)} days")
    except Exception as e:
        print(f"Failed to load {symbol}: {e}")

## 3. Compute Latent Knowledge Features

In [ ]:
# Initialize feature computers
lkf = LatentKnowledgeFeatures()

# Compute features for each symbol
features_by_symbol = {}

for symbol, df in price_data.items():
    try:
        features = lkf.compute_all(df, symbol=symbol)
        features_by_symbol[symbol] = features
        print(f"{symbol}: {len(features.columns)} features computed")
    except Exception as e:
        print(f"Failed to compute features for {symbol}: {e}")

# Show feature columns
if features_by_symbol:
    sample_features = list(features_by_symbol.values())[0]
    print(f"\nFeature columns ({len(sample_features.columns)}):")
    for col in sample_features.columns:
        print(f"  - {col}")

## 4. Hypothesis Testing: Information Coefficient

In [ ]:
def compute_feature_ic(features, returns, horizons=[1, 5, 20]):
    """Compute IC for each feature against forward returns."""
    results = []
    
    for col in features.columns:
        feature_values = features[col].dropna()
        
        for horizon in horizons:
            # Align with forward returns
            fwd_returns = returns.shift(-horizon)
            aligned = pd.concat([feature_values, fwd_returns], axis=1).dropna()
            
            if len(aligned) < 30:
                continue
            
            # Compute rank correlation (IC)
            ic, p_value = stats.spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])
            
            results.append({
                'feature': col,
                'horizon': horizon,
                'ic': ic,
                'p_value': p_value,
                'n_samples': len(aligned),
                'significant': p_value < 0.05,
            })
    
    return pd.DataFrame(results)

In [ ]:
# Compute IC for all features across all symbols
all_ic_results = []

for symbol, features in features_by_symbol.items():
    if symbol not in price_data:
        continue
    
    returns = price_data[symbol]['close'].pct_change()
    
    ic_df = compute_feature_ic(features, returns)
    ic_df['symbol'] = symbol
    all_ic_results.append(ic_df)

if all_ic_results:
    ic_results = pd.concat(all_ic_results, ignore_index=True)
    print(f"Computed IC for {len(ic_results)} feature-horizon-symbol combinations")
else:
    ic_results = pd.DataFrame()

In [ ]:
# Aggregate IC across symbols
if len(ic_results) > 0:
    feature_ic_summary = ic_results.groupby(['feature', 'horizon']).agg({
        'ic': ['mean', 'std'],
        'significant': 'mean',
        'symbol': 'count',
    })
    feature_ic_summary.columns = ['mean_ic', 'std_ic', 'pct_significant', 'n_symbols']
    feature_ic_summary = feature_ic_summary.reset_index()
    
    # Best features at 5-day horizon
    best_5d = feature_ic_summary[feature_ic_summary['horizon'] == 5].sort_values(
        'mean_ic', ascending=False
    )
    
    print("Top 15 Features by 5-Day IC:")
    display(best_5d.head(15))

## 5. Hypothesis Validation Results

In [ ]:
# Map hypotheses to features
HYPOTHESIS_FEATURE_MAP = {
    'gap_fade': ['gap_fade_signal', 'gap_faded', 'large_gap'],
    'monday_reversal': ['monday_reversal_signal', 'is_monday', 'friday_momentum'],
    'vix_mean_reversion': ['vix_mean_reversion_signal', 'vix_extreme_high', 'post_spike_signal'],
    'month_end_drift': ['is_month_end', 'month_end_effect', 'days_to_month_end'],
    'opex_effects': ['is_opex_week'],
    'vol_clustering': ['vol_expansion', 'vol_regime', 'vol_percentile'],
    'post_earnings_drift': ['post_earnings_drift_signal', 'last_earnings_reaction'],
    'credit_leads_equity': ['credit_stress_signal', 'credit_widening'],
    'banks_rates': ['rising_rates_signal', 'rates_beta_20d'],
    'sector_lead_lag': ['sector_lead_signal', 'leader_momentum', 'lead_lag_divergence'],
}

# Validate each hypothesis
validation_results = []

if len(ic_results) > 0:
    for hypothesis, features in HYPOTHESIS_FEATURE_MAP.items():
        hypothesis_ic = ic_results[
            (ic_results['feature'].isin(features)) &
            (ic_results['horizon'] == 5)
        ]
        
        if len(hypothesis_ic) > 0:
            mean_ic = hypothesis_ic['ic'].mean()
            pct_sig = hypothesis_ic['significant'].mean()
            best_feature = hypothesis_ic.loc[hypothesis_ic['ic'].abs().idxmax()]
            
            validation_results.append({
                'hypothesis': hypothesis,
                'description': hypotheses.get(hypothesis, 'N/A'),
                'mean_ic': mean_ic,
                'pct_significant': pct_sig,
                'best_feature': best_feature['feature'],
                'best_ic': best_feature['ic'],
                'status': 'VALIDATED' if abs(mean_ic) > 0.02 and pct_sig > 0.3 else (
                    'WEAK' if abs(mean_ic) > 0.01 else 'NOT VALIDATED'
                )
            })
        else:
            validation_results.append({
                'hypothesis': hypothesis,
                'description': hypotheses.get(hypothesis, 'N/A'),
                'mean_ic': np.nan,
                'pct_significant': np.nan,
                'best_feature': 'N/A',
                'best_ic': np.nan,
                'status': 'NO DATA'
            })

validation_df = pd.DataFrame(validation_results)

print("=== Hypothesis Validation Results ===")
display(validation_df)

In [ ]:
# Visualize validation results
if len(validation_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    colors = {
        'VALIDATED': 'green',
        'WEAK': 'yellow',
        'NOT VALIDATED': 'red',
        'NO DATA': 'gray',
    }
    
    valid_df = validation_df.dropna(subset=['mean_ic'])
    if len(valid_df) > 0:
        bar_colors = [colors[s] for s in valid_df['status']]
        
        ax.barh(range(len(valid_df)), valid_df['mean_ic'], color=bar_colors)
        ax.set_yticks(range(len(valid_df)))
        ax.set_yticklabels(valid_df['hypothesis'])
        ax.axvline(0.02, color='green', linestyle='--', label='IC=0.02 (good)')
        ax.axvline(-0.02, color='green', linestyle='--')
        ax.axvline(0, color='gray', linestyle='-', alpha=0.5)
        ax.set_xlabel('Mean IC (5-day horizon)')
        ax.set_title('Latent Knowledge Hypothesis Validation')
        ax.legend()
        
    plt.tight_layout()
    plt.show()

## 6. Regime-Conditional Analysis

In [ ]:
# Analyze IC by volatility regime
if 'SPY' in price_data and 'SPY' in features_by_symbol:
    spy_features = features_by_symbol['SPY']
    spy_returns = price_data['SPY']['close'].pct_change()
    
    # Classify volatility regime
    vol_20d = spy_returns.rolling(20).std() * np.sqrt(252)
    vol_percentile = vol_20d.rolling(252).rank(pct=True)
    
    # Define regimes
    low_vol = vol_percentile < 0.25
    high_vol = vol_percentile > 0.75
    normal_vol = ~low_vol & ~high_vol
    
    # Compute IC for each regime
    regime_results = []
    
    for regime_name, regime_mask in [('Low Vol', low_vol), ('Normal', normal_vol), ('High Vol', high_vol)]:
        regime_features = spy_features[regime_mask]
        regime_returns = spy_returns[regime_mask]
        
        if len(regime_features) > 50:
            ic_df = compute_feature_ic(regime_features, regime_returns, horizons=[5])
            ic_df['regime'] = regime_name
            regime_results.append(ic_df)
    
    if regime_results:
        regime_ic = pd.concat(regime_results, ignore_index=True)
        
        # Compare regimes
        regime_comparison = regime_ic.pivot_table(
            values='ic',
            index='feature',
            columns='regime',
            aggfunc='mean'
        )
        
        print("Top Features with Regime-Dependent IC:")
        regime_comparison['max_diff'] = regime_comparison.max(axis=1) - regime_comparison.min(axis=1)
        display(regime_comparison.sort_values('max_diff', ascending=False).head(15))

## 7. Summary and Recommendations

In [ ]:
print("=" * 70)
print("LATENT KNOWLEDGE VALIDATION SUMMARY")
print("=" * 70)

if len(validation_df) > 0:
    validated = validation_df[validation_df['status'] == 'VALIDATED']
    weak = validation_df[validation_df['status'] == 'WEAK']
    not_validated = validation_df[validation_df['status'] == 'NOT VALIDATED']
    
    print(f"\nHypotheses tested: {len(validation_df)}")
    print(f"Validated (IC > 0.02, >30% significant): {len(validated)}")
    print(f"Weak signal (IC > 0.01): {len(weak)}")
    print(f"Not validated: {len(not_validated)}")
    
    print("\n" + "-" * 70)
    print("VALIDATED HYPOTHESES (Use in production):")
    print("-" * 70)
    for _, row in validated.iterrows():
        print(f"\n✓ {row['hypothesis']}")
        print(f"  {row['description']}")
        print(f"  Best feature: {row['best_feature']} (IC={row['best_ic']:.3f})")
    
    print("\n" + "-" * 70)
    print("WEAK HYPOTHESES (Need more data or refinement):")
    print("-" * 70)
    for _, row in weak.iterrows():
        print(f"\n? {row['hypothesis']}")
        print(f"  {row['description']}")
        print(f"  IC={row['mean_ic']:.3f} - Consider refining or acquiring more data")
    
    print("\n" + "-" * 70)
    print("NOT VALIDATED (Re-evaluate or discard):")
    print("-" * 70)
    for _, row in not_validated.iterrows():
        print(f"\n✗ {row['hypothesis']}")
        print(f"  {row['description']}")
else:
    print("No validation results available.")

In [ ]:
print("\n" + "=" * 70)
print("DATA ACQUISITION RECOMMENDATIONS")
print("=" * 70)

recommendations = [
    {
        'hypothesis': 'credit_leads_equity',
        'needed_data': 'Credit spread data (HY spreads, CDS)',
        'expected_ic': 0.03,
        'priority': 'HIGH',
    },
    {
        'hypothesis': 'post_earnings_drift',
        'needed_data': 'Historical earnings surprises and reactions',
        'expected_ic': 0.04,
        'priority': 'HIGH',
    },
    {
        'hypothesis': 'insider_timing',
        'needed_data': 'Full SEC Form 4 filing database',
        'expected_ic': 0.03,
        'priority': 'MEDIUM',
    },
    {
        'hypothesis': 'options_flow_lead',
        'needed_data': 'Real-time options flow with block trades',
        'expected_ic': 0.025,
        'priority': 'MEDIUM',
    },
    {
        'hypothesis': 'analyst_herding',
        'needed_data': 'Analyst estimate revisions history',
        'expected_ic': 0.02,
        'priority': 'LOW',
    },
]

for rec in recommendations:
    print(f"\n[{rec['priority']}] {rec['hypothesis']}")
    print(f"  Data needed: {rec['needed_data']}")
    print(f"  Expected IC: {rec['expected_ic']}")